# Round20 全流程 Notebook：`v20_4_ta_ec_struct_farcal_alt.csv`

这个 notebook 是完整复现流程：从官方 GitHub 原始数据读取，到结构化特征建模、策略渲染、导出提交文件、诊断分析。

**当前分支配置**
- label: `ta_ec_struct_farcal_alt`
- branch: `ta_ec_struct_farcal_alt`
- theme: `TA static-geo + EC farcal alt`
- risk_level: `medium-high`


## Step 1. Experiment Objective

**这个步骤做什么**
明确 round20 分支在“稳定复现 + 结构探索”框架中的定位。

**为什么要做这个步骤**
确保每个提交文件都提供不同的信息增益，而不是重复微调。

**预期产出**
得到本分支的实验角色定义。

## Step 2. Environment Setup

**这个步骤做什么**
导入依赖并初始化路径与数据源。

**为什么要做这个步骤**
保证 notebook 可复现可迁移。

**预期产出**
得到统一运行环境。

In [ ]:
# =========================
# Step: Environment Setup
# 作用：导入依赖，定义路径和官方数据源
# =========================

import json
import subprocess
from io import StringIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsRegressor, NearestNeighbors

pd.set_option('display.max_columns', 260)
pd.set_option('display.width', 220)

PROJECT_ROOT = Path('/Users/zhangziling/Documents/New project')
ROUND_DIR = PROJECT_ROOT / 'round20'
BASE_RAW = 'https://raw.githubusercontent.com/Snowflake-Labs/EY-AI-and-Data-Challenge/refs/heads/main'

## Step 3. Load Official Raw Data

**这个步骤做什么**
从官方 GitHub 读取训练/验证原始数据。

**为什么要做这个步骤**
保证数据来源一致且可审计。

**预期产出**
得到 6 张官方原始表。

In [ ]:
# =========================
# Step: Load Official Raw Data From GitHub
# 作用：读取挑战赛官方原始训练/验证数据
# =========================

def read_csv_github(url: str) -> pd.DataFrame:
    try:
        return pd.read_csv(url)
    except Exception as e:
        print(f'[warn] pd.read_csv failed, fallback to curl: {url}')
        print(f'       reason: {e}')
        res = subprocess.run(['curl', '-L', url], check=True, capture_output=True, text=True)
        return pd.read_csv(StringIO(res.stdout))

train_wq = read_csv_github(f'{BASE_RAW}/water_quality_training_dataset.csv')
train_ls = read_csv_github(f'{BASE_RAW}/landsat_features_training.csv')
train_tc = read_csv_github(f'{BASE_RAW}/terraclimate_features_training.csv')

val_tpl = read_csv_github(f'{BASE_RAW}/submission_template.csv')
val_ls = read_csv_github(f'{BASE_RAW}/landsat_features_validation.csv')
val_tc = read_csv_github(f'{BASE_RAW}/terraclimate_features_validation.csv')

print('train_wq:', train_wq.shape)
print('train_ls:', train_ls.shape)
print('train_tc:', train_tc.shape)
print('val_tpl:', val_tpl.shape)
print('val_ls:', val_ls.shape)
print('val_tc:', val_tc.shape)

## Step 4. Data QA

**这个步骤做什么**
检查形状、字段和缺失率。

**为什么要做这个步骤**
提前暴露数据问题，避免后续误判。

**预期产出**
得到基础 QA 报告。

In [ ]:
# =========================
# Step: Data QA
# 作用：检查数据结构和缺失率
# =========================

def qa_table(df: pd.DataFrame, name: str, topn: int = 10):
    print(f'\n===== {name} =====')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    miss = df.isna().mean().sort_values(ascending=False).head(topn)
    print('top missing ratio:')
    print(miss)

qa_table(train_wq, 'train_wq')
qa_table(train_ls, 'train_ls')
qa_table(train_tc, 'train_tc')
qa_table(val_tpl, 'val_tpl')

display(train_wq.head(3))
display(val_tpl.head(3))

## Step 5. Load Anchors

**这个步骤做什么**
加载 v2、control、replay 三个锚点文件。

**为什么要做这个步骤**
round20 采用“以 TA replay 为主轴 + 结构微扰”的路线，需要锚点做结果归因。

**预期产出**
得到可用于对照的历史锚点数组。

In [ ]:
# =========================
# Step: Load Anchor Files
# 作用：读取 v2、control、replay（三个锚点），保证策略渲染可对齐历史最佳路线
# =========================

def load_existing_submission(candidates):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return pd.read_csv(p), p
    raise FileNotFoundError(f'None of candidate files exists: {candidates}')

v2_ref = pd.read_csv(PROJECT_ROOT / 'submission_template_v2.csv')
control_ref = pd.read_csv(PROJECT_ROOT / 'round15/v15_3_ta_refine_push.csv')

replay_ref, replay_path = load_existing_submission([
    PROJECT_ROOT / 'round19/v19_1_ta_core_replay.csv',
    PROJECT_ROOT / 'round18/v18_3_ta_core_center.csv',
    PROJECT_ROOT / 'round18/18_3_ta_core_center.csv',
])

assert len(v2_ref) == len(val_tpl), 'v2_ref length mismatch'
assert len(control_ref) == len(val_tpl), 'control_ref length mismatch'
assert len(replay_ref) == len(val_tpl), 'replay_ref length mismatch'

print('anchor files loaded:')
print('  v2:', PROJECT_ROOT / 'submission_template_v2.csv')
print('  control:', PROJECT_ROOT / 'round15/v15_3_ta_refine_push.csv')
print('  replay:', replay_path)

## Step 6. Define Structural Helpers

**这个步骤做什么**
定义 TA 静态地理模型和 DRP 结构模型函数。

**为什么要做这个步骤**
让每个结构分支可复现、可解释。

**预期产出**
得到 round20 核心函数库。

In [ ]:
# =========================
# Step: Define Strategy Helper Functions
# 作用：定义 round20 的结构化建模函数（TA静态地理 + DRP新模型）
# =========================

def compute_scale(train_coords: pd.DataFrame, val_coords: pd.DataFrame) -> np.ndarray:
    tr_unique = train_coords[['Latitude', 'Longitude']].drop_duplicates().to_numpy()
    va = val_coords[['Latitude', 'Longitude']].to_numpy()
    nn = NearestNeighbors(n_neighbors=1, metric='euclidean').fit(tr_unique)
    dist, _ = nn.kneighbors(va)
    dist = dist.reshape(-1)
    q10, q90 = np.quantile(dist, [0.10, 0.90])
    return np.clip((dist - q10) / max(q90 - q10, 1e-9), 0.0, 1.0)


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out['Sample Date'], dayfirst=True, errors='coerce')
    out['month'] = dt.dt.month.astype(float)
    out['dayofyear'] = dt.dt.dayofyear.astype(float)
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12.0)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12.0)
    out['doy_sin'] = np.sin(2 * np.pi * out['dayofyear'] / 366.0)
    out['doy_cos'] = np.cos(2 * np.pi * out['dayofyear'] / 366.0)
    return out


def build_ta_geo_prediction(train_wq: pd.DataFrame, val_tpl: pd.DataFrame) -> np.ndarray:
    tr = add_time_features(train_wq)
    va = add_time_features(val_tpl)
    feat_cols = ['Latitude', 'Longitude', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos']
    xtr = tr[feat_cols].to_numpy(dtype=float)
    xva = va[feat_cols].to_numpy(dtype=float)
    y = tr['Total Alkalinity'].to_numpy(dtype=float)

    model = KNeighborsRegressor(n_neighbors=20, weights='distance', p=2)
    model.fit(xtr, y)
    pred = model.predict(xva)
    return np.clip(pred, 0, None)


def build_drp_geo_prediction(train_wq, train_ls, train_tc, val_tpl, val_ls, val_tc):
    key = ['Latitude', 'Longitude', 'Sample Date']
    for df in [train_wq, train_ls, train_tc, val_tpl, val_ls, val_tc]:
        df['Sample Date'] = pd.to_datetime(df['Sample Date'], dayfirst=True, errors='coerce')

    train_full = train_wq.merge(train_ls, on=key, how='left', suffixes=('', '_ls')).merge(train_tc, on=key, how='left', suffixes=('', '_tc'))
    val_full = val_tpl.merge(val_ls, on=key, how='left', suffixes=('', '_ls')).merge(val_tc, on=key, how='left', suffixes=('', '_tc'))

    def add_features(df):
        out = add_time_features(df)
        out['lat_abs'] = out['Latitude'].abs()
        out['lon_abs'] = out['Longitude'].abs()
        out['lat_x_lon'] = out['Latitude'] * out['Longitude']
        out['lat2'] = out['Latitude'] ** 2
        out['lon2'] = out['Longitude'] ** 2
        eps = 1e-6
        if {'nir', 'green'}.issubset(out.columns):
            out['ndvi'] = (out['nir'] - out['green']) / (out['nir'] + out['green'] + eps)
        if {'swir16', 'nir'}.issubset(out.columns):
            out['ndbi'] = (out['swir16'] - out['nir']) / (out['swir16'] + out['nir'] + eps)
        if {'swir22', 'swir16'}.issubset(out.columns):
            out['swir_ratio'] = out['swir22'] / (out['swir16'] + eps)
        if {'pet', 'NDMI'}.issubset(out.columns):
            out['pet_x_ndmi'] = out['pet'] * out['NDMI']
        if {'pet', 'MNDWI'}.issubset(out.columns):
            out['pet_x_mndwi'] = out['pet'] * out['MNDWI']
        return out

    tr = add_features(train_full)
    va = add_features(val_full)

    ignore = set(key + ['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
    feat_cols = [c for c in tr.columns if c in va.columns and c not in ignore]
    xtr = tr[feat_cols]
    xva = va[feat_cols]
    y = np.log1p(np.clip(tr['Dissolved Reactive Phosphorus'].to_numpy(dtype=float), 0, None))

    imp = SimpleImputer(strategy='median')
    xtr_i = imp.fit_transform(xtr)
    xva_i = imp.transform(xva)

    model = HistGradientBoostingRegressor(
        learning_rate=0.035,
        max_depth=6,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=0.2,
        max_iter=700,
        random_state=42,
    )
    model.fit(xtr_i, y)

    pred = np.expm1(model.predict(xva_i))
    return np.clip(pred, 0, None)


def ec_farcal(control_ec, v2_ec, scale):
    w = 0.01 + 0.035 * np.clip((scale - 0.76) / 0.24, 0, 1)
    out = (1.0 - w) * control_ec + w * v2_ec
    return np.clip(out, 0, None)


def ec_farcal_alt(control_ec, v2_ec, scale):
    w = 0.016 + 0.040 * np.clip((scale - 0.72) / 0.28, 0, 1)
    out = (1.0 - w) * control_ec + w * v2_ec
    return np.clip(out, 0, None)


def ta_geo_blend(replay_ta, ta_geo, scale, max_w, near_power, delta_clip_abs):
    delta = np.clip(ta_geo - replay_ta, -delta_clip_abs, delta_clip_abs)
    w = max_w * np.power(1.0 - scale, near_power)
    out = replay_ta + w * delta
    return np.clip(out, 0, None)


def drp_geo_blend(control_drp, drp_geo_pred, scale, max_w, near_power, delta_clip_abs):
    delta = np.clip(drp_geo_pred - control_drp, -delta_clip_abs, delta_clip_abs)
    w = max_w * np.power(1.0 - scale, near_power)
    out = control_drp + w * delta
    # 避免分位数截断对全局分布引入“无意偏移”，尤其在超轻微扰动分支中。
    return np.clip(out, 0, None)

## Step 7. Build Structural Signals

**这个步骤做什么**
训练并生成 TA/DRP 的结构化预测信号。

**为什么要做这个步骤**
这是本轮突破平台的关键探索信号来源。

**预期产出**
得到 TA geo（mild/mid/ec）/ DRP ultra / EC farcal alt 等中间数组。

In [ ]:
# =========================
# Step: Build Structural Signals
# 作用：生成 TA 静态地理预测、DRP 结构模型预测及各类可控融合信号
# =========================

scale_val = compute_scale(train_wq[['Latitude', 'Longitude']], val_tpl[['Latitude', 'Longitude']])

control_ta = control_ref['Total Alkalinity'].to_numpy(dtype=float)
control_ec = control_ref['Electrical Conductance'].to_numpy(dtype=float)
control_drp = control_ref['Dissolved Reactive Phosphorus'].to_numpy(dtype=float)

replay_ta = replay_ref['Total Alkalinity'].to_numpy(dtype=float)
v2_ec = v2_ref['Electrical Conductance'].to_numpy(dtype=float)

ta_geo_pred = build_ta_geo_prediction(train_wq.copy(), val_tpl.copy())
drp_geo_pred = build_drp_geo_prediction(
    train_wq.copy(), train_ls.copy(), train_tc.copy(),
    val_tpl.copy(), val_ls.copy(), val_tc.copy()
)

ec_far = ec_farcal(control_ec=control_ec, v2_ec=v2_ec, scale=scale_val)
ec_far_alt = ec_farcal_alt(control_ec=control_ec, v2_ec=v2_ec, scale=scale_val)

ta_geo_mild = ta_geo_blend(
    replay_ta=replay_ta, ta_geo=ta_geo_pred, scale=scale_val,
    max_w=0.010, near_power=1.40, delta_clip_abs=24.0
)
ta_geo_mid = ta_geo_blend(
    replay_ta=replay_ta, ta_geo=ta_geo_pred, scale=scale_val,
    max_w=0.016, near_power=1.25, delta_clip_abs=24.0
)
ta_geo_ec = ta_geo_blend(
    replay_ta=replay_ta, ta_geo=ta_geo_pred, scale=scale_val,
    max_w=0.014, near_power=1.10, delta_clip_abs=24.0
)

drp_geo_ultra = drp_geo_blend(
    control_drp=control_drp, drp_geo_pred=drp_geo_pred, scale=scale_val,
    max_w=0.0016, near_power=2.10, delta_clip_abs=0.55
)

print('signals ready.')
print('scale min/max:', float(scale_val.min()), float(scale_val.max()))
print('mean |TA geo - replay|:', float(np.mean(np.abs(ta_geo_pred - replay_ta))))
print('mean |DRP geo - control|:', float(np.mean(np.abs(drp_geo_pred - control_drp))))

## Step 8. Render Branch And Export

**这个步骤做什么**
按当前分支配置渲染并导出提交文件。

**为什么要做这个步骤**
确保每个文件可独立重跑、独立上传。

**预期产出**
得到 `v20_4_ta_ec_struct_farcal_alt.csv`。

In [ ]:
# =========================
# Step: Render Branch And Export CSV
# 作用：根据 branch 配置渲染预测并导出提交文件
# =========================

cfg = json.loads(r'''{
  "file_name": "v20_4_ta_ec_struct_farcal_alt.csv",
  "label": "ta_ec_struct_farcal_alt",
  "theme": "TA static-geo + EC farcal alt",
  "risk_level": "medium-high",
  "branch": "ta_ec_struct_farcal_alt",
  "ta_model": "knn_geo_time",
  "ta_blend_max_w": 0.014,
  "ta_near_power": 1.1,
  "ta_delta_clip_abs": 24.0,
  "ec_mode": "farcal_alt",
  "ta_shift_vs_v2": 13.151914894984397,
  "ec_shift_vs_v2": 100.7587965412529,
  "drp_shift_vs_v2": 0.0,
  "ta_shift_vs_replay": 0.12498491591212701,
  "ec_shift_vs_replay": 2.398367015567155,
  "drp_shift_vs_replay": 0.0,
  "ta_shift_vs_control": 2.0725916465355123,
  "ec_shift_vs_control": 2.398367015567155,
  "drp_shift_vs_control": 0.0
}''')
branch = cfg.get('branch')

print('branch:', branch)
print('label:', cfg.get('label'))
print('theme:', cfg.get('theme'))

if branch == 'ta_replay':
    ta = replay_ta.copy()
    ec = control_ec.copy()
    drp = control_drp.copy()
elif branch == 'ta_static_geo_mild':
    ta = ta_geo_mild.copy()
    ec = control_ec.copy()
    drp = control_drp.copy()
elif branch == 'ta_static_geo_mid':
    ta = ta_geo_mid.copy()
    ec = control_ec.copy()
    drp = control_drp.copy()
elif branch == 'ta_ec_struct_farcal_alt':
    ta = ta_geo_ec.copy()
    ec = ec_far_alt.copy()
    drp = control_drp.copy()
elif branch == 'ta_drp_ultralight_cap':
    ta = replay_ta.copy()
    ec = control_ec.copy()
    drp = drp_geo_ultra.copy()
else:
    raise ValueError(f'Unknown branch: {branch}')

sub = val_tpl[['Latitude', 'Longitude', 'Sample Date']].copy()
sub['Sample Date'] = pd.to_datetime(sub['Sample Date'], dayfirst=True, errors='coerce').dt.strftime('%d/%m/%Y')
sub['Total Alkalinity'] = np.clip(ta, 0, None)
sub['Electrical Conductance'] = np.clip(ec, 0, None)
sub['Dissolved Reactive Phosphorus'] = np.clip(drp, 0, None)
sub = sub[['Latitude', 'Longitude', 'Sample Date', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus']]

out_file = ROUND_DIR / 'v20_4_ta_ec_struct_farcal_alt.csv'
sub.to_csv(out_file, index=False)
print('saved:', out_file)
print('shape:', sub.shape)

diag = {
    'ta_shift_vs_replay': float(np.mean(np.abs(sub['Total Alkalinity'] - replay_ref['Total Alkalinity']))),
    'ec_shift_vs_replay': float(np.mean(np.abs(sub['Electrical Conductance'] - replay_ref['Electrical Conductance']))),
    'drp_shift_vs_replay': float(np.mean(np.abs(sub['Dissolved Reactive Phosphorus'] - replay_ref['Dissolved Reactive Phosphorus']))),
    'ta_shift_vs_control': float(np.mean(np.abs(sub['Total Alkalinity'] - control_ref['Total Alkalinity']))),
    'ec_shift_vs_control': float(np.mean(np.abs(sub['Electrical Conductance'] - control_ref['Electrical Conductance']))),
    'drp_shift_vs_control': float(np.mean(np.abs(sub['Dissolved Reactive Phosphorus'] - control_ref['Dissolved Reactive Phosphorus']))),
}

print('diagnostics:')
for k, v in diag.items():
    print(f'  - {k}: {v:.6f}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(sub['Total Alkalinity'], bins=45, color='steelblue', alpha=0.85)
axes[0].set_title('TA Distribution')
axes[1].hist(sub['Electrical Conductance'], bins=45, color='darkorange', alpha=0.85)
axes[1].set_title('EC Distribution')
axes[2].hist(sub['Dissolved Reactive Phosphorus'], bins=45, color='seagreen', alpha=0.85)
axes[2].set_title('DRP Distribution')
plt.tight_layout()
plt.show()

sub.head()

## Step 9. Result Interpretation

**这个步骤做什么**
解释本分支在“低风险复现/结构探索”框架中的定位。

**为什么要做这个步骤**
每天提交机会有限，需要确保每个文件承担明确的实验角色。

**本分支摘要**
- 输出文件：`v20_4_ta_ec_struct_farcal_alt.csv`
- label：`ta_ec_struct_farcal_alt`
- branch：`ta_ec_struct_farcal_alt`
- theme：`TA static-geo + EC farcal alt`
- 重点看：`ta_shift_vs_replay`、`drp_shift_vs_control`、分布稳定性。